# Customer Intelligence & Support Copilot
## Instructor-led end-to-end notebook

**Scenario:** synthetic grocery quick-commerce data. This notebook emphasizes *how data is fetched, inspected, cleaned, validated, stored, and then consumed by downstream AI/ML components*.

### Before class
1. Run `python generate_data.py` from the repository root.
2. Install packages with `pip install -r requirements.txt`.
3. Run this notebook from the repository root so relative paths work.

All data and policies are fictional. The LLM calls are handled by `mock_llm.py` locally; no API key is required.

## 0. Repository map and end-to-end flow

`generate_data.py` → `data/raw/*.csv` → audit → clean → `data/clean/*.csv` + `data/instamart.db` → RFM/segments + lapse model → privacy-safe ticket triage → policy retrieval + refund tools.

Teaching point: **fetching data** does not always mean calling an API. Here, the source is a local CSV export. In a production system, the same ingestion layer could use an approved API or warehouse connector, but credentials and network access are intentionally out of scope.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: c:\Users\HP\Documents\Instamart customer copilot\instamart_customer_copilot_teaching_kit
Root files: ['.gitignore', 'GITHUB_RESOURCES.md', 'README.md', 'generate_data.py', 'mock_llm.py', 'requirements.txt', 'run_project.py']


## 1. Generate / fetch the raw data

The data generator is deterministic (fixed random seed). It writes customer, product, order, support-ticket, evaluation-question, agent-scenario inputs and policy text files.

Run it once when starting from a clean clone. Running it again recreates the same synthetic source exports.

In [16]:
import subprocess, sys
# ROOT = Path(r"c:\Users\HP\Documents\Instamart customer copilot\instamart_customer_copilot_teaching_kit")
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

Raw files: ['customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


## 2. Read the exports and understand their shape

Use `keep_default_na=False` for ticket text so empty strings remain empty strings. This is important because empty tickets must be identified by an explicit guardrail rather than silently changing the input representation.

In [17]:
import pandas as pd
customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [("customers", customers_raw), ("products", products_raw),
                 ("orders", orders_raw), ("tickets", tickets_raw)]:
    print(f"\\n{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(lambda col: col.str.strip().eq("").sum()),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }))

\ncustomers: shape=(1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F


,dtype,missing_count,blank_count,missing_pct
customer_id,str,0,0,0.0
city,str,0,0,0.0
signup_date,str,0,0,0.0
age,str,0,61,0.0
gender,str,0,30,0.0


\nproducts: shape=(40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


,dtype,missing_count,blank_count,missing_pct
product_id,str,0,0,0.0
product_name,str,0,0,0.0
category,str,0,0,0.0
price_inr,int64,0,0,0.0


... (2 outputs left out)

,dtype,missing_count,blank_count,missing_pct
order_id,str,0,0,0.0
customer_id,str,0,0,0.0
product_id,str,0,0,0.0
quantity,int64,0,0,0.0
order_ts,str,0,0,0.0
status,str,0,0,0.0
discount_pct,int64,0,0,0.0
order_amount,str,0,0,0.0
delivery_minutes,str,0,2541,0.0


\ntickets: shape=(164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


,dtype,missing_count,blank_count,missing_pct
ticket_id,str,0,0,0.0
customer_id,str,0,0,0.0
order_id,str,0,8,0.0
text,str,0,4,0.0
true_category,str,0,0,0.0


### Audit questions for students
- Which columns are supposed to be unique keys?
- Are duplicates exact duplicates or multiple states of the same order?
- Which values are impossible (negative delivery duration, non-numeric money, implausible age)?
- Why should we distinguish an empty string from a missing value?
- What would go wrong if we modelled before cleaning?

In [18]:
from src.data_pipeline import audit, clean_data, validate
audit()


customers: shape=(1200, 5)
customer_id    str
city           str
signup_date    str
age            str
gender         str
dtype: object
             missing_count  missing_pct
customer_id              0          0.0
city                     0          0.0
signup_date              0          0.0
age                      0          0.0
gender                   0          0.0

products: shape=(40, 4)
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

orders: shape=(13084, 9)
order_id              str
customer_id           str
product_id            str
quantity            int64
order_ts              str
status                str
discount_pct        int64
order_amount          str
delivery_minutes      str
dtype: object
                 

## 3. Clean the data and write trusted outputs

The pipeline applies explicit rules:
- standardize city and gender;
- make impossible ages missing and fill with city median;
- parse timestamps, remove exact duplicates, then keep the latest row per order ID;
- coerce invalid amounts to missing and calculate replacements from product price × quantity × discount;
- set negative delivery times to missing and impute only completed orders using product-category medians;
- save clean CSVs and SQLite tables.

Ask students to explain why cancelled orders should keep a missing delivery duration.

In [21]:
clean = clean_data()
customers = clean["customers"]
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
validate()


customers (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35.0,MALE
1,C0002,DELHI,2024-02-17,31.0,MALE
2,C0003,bengaluru,2025-02-18,18.0,F


Customer data validation
Missing values:
customer_id     0
city            0
signup_date     0
age            61
gender         30
dtype: int64

Duplicate rows:
0

Dataset shape:
(1200, 5)


In [22]:
import numpy as np
import pandas as pd


customers = pd.read_csv("data/raw/customers.csv")
orders = pd.read_csv("data/raw/orders.csv")
products = pd.read_csv("data/raw/products.csv")
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Products:", products.shape)
def clean_customers(customers):

    customers = customers.copy()

    # (a) Standardise city names
    customers["city"] = (
        customers["city"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    # (b) Standardise gender
    gender_map = {
        "male": "male",
        "m": "male",
        "man": "male",
        "female": "female",
        "f": "female",
        "woman": "female"
    }

    customers["gender"] = (
        customers["gender"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map(gender_map)
        .fillna("unknown")
    )

    # (c) Convert age to numeric
    customers["age"] = pd.to_numeric(
        customers["age"], errors="coerce"
    )

    # Treat ages below 10 or above 100 as missing
    customers.loc[
        (customers["age"] < 10) |
        (customers["age"] > 100),
        "age"
    ] = np.nan

    # Fill missing ages using city median
    city_median = (
        customers.groupby("city")["age"]
        .transform("median")
    )

    customers["age"] = customers["age"].fillna(
        city_median
    )

    # (d) Convert signup_date to datetime
    customers["signup_date"] = pd.to_datetime(
        customers["signup_date"],
        errors="coerce"
    )
    return customers
customers = pd.read_csv("data/raw/customers.csv")
customers = clean_customers(customers)

print("Customer cleaning completed!")
print("Shape:", customers.shape)
display(customers.head())
print("CITY COUNTS")
print(customers["city"].value_counts())

print("\nGENDER COUNTS")
print(customers["gender"].value_counts())

print("\nMISSING AGES")
print(customers["age"].isna().sum())

print("\nAGE RANGE")
print(customers["age"].min(), customers["age"].max())

print("\nSIGNUP DATE TYPE")
print(customers["signup_date"].dtype)

def clean_orders(orders, products):

    orders = orders.copy()
    products = products.copy()

    # 1. Convert timestamps
    orders["order_ts"] = pd.to_datetime(
        orders["order_ts"], errors="coerce"
    )

    # 2. Remove exact duplicate rows
    orders = orders.drop_duplicates()

    # 3. Keep latest record for each order_id
    orders = (
        orders.sort_values("order_ts", kind="stable")
        .drop_duplicates("order_id", keep="last")
        .copy()
    )

    # 4. Merge product information using product_id
    product_lookup = products[
        ["product_id", "category", "price_inr"]
    ].drop_duplicates("product_id")

    orders = orders.merge(
        product_lookup,
        on="product_id",
        how="left",
        validate="many_to_one"
    )

    # Convert numeric columns
    for col in ["order_amount", "quantity",
                "discount_pct", "price_inr"]:
        orders[col] = pd.to_numeric(
            orders[col], errors="coerce"
        )

    # Calculate missing amounts
    calculated_amount = (
        orders["price_inr"]
        * orders["quantity"]
        * (1 - orders["discount_pct"] / 100)
    ).round(2)

    orders["order_amount"] = (
        orders["order_amount"].fillna(calculated_amount)
    )

    # 5. Clean delivery times
    orders["delivery_minutes"] = pd.to_numeric(
        orders["delivery_minutes"], errors="coerce"
    )

    orders.loc[
        orders["delivery_minutes"] < 0,
        "delivery_minutes"
    ] = np.nan

    # Impute only completed orders
    completed = orders["status"].eq("completed")

    category_median = (
        orders.loc[completed]
        .groupby("category")["delivery_minutes"]
        .transform("median")
    )

    orders.loc[completed, "delivery_minutes"] = (
        orders.loc[completed, "delivery_minutes"]
        .fillna(category_median)
    )

    return orders
customers = clean_customers(customers)

orders = clean_orders(orders, products)

print("Customers:", customers.shape)
print("Orders:", orders.shape)

print("\nORDER STATUS:")
print(orders["status"].value_counts())

print("\nDUPLICATE ORDER IDs:")
print(orders["order_id"].duplicated().sum())

print("\nMISSING ORDER AMOUNTS:")
print(orders["order_amount"].isna().sum())

print("\nMISSING DELIVERY TIMES:")
print(orders.groupby("status")["delivery_minutes"].apply(
    lambda x: x.isna().sum()
))

print("\nCOMPLETED ORDER AMOUNT:")
print(round(
    orders.loc[
        orders["status"] == "completed",
        "order_amount"
    ].sum(), 2
))




Customers: (1200, 5)
Orders: (13084, 9)
Products: (40, 4)
Customer cleaning completed!
Shape: (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35.0,male
1,C0002,Delhi,2024-02-17,31.0,male
2,C0003,Bengaluru,2025-02-18,18.0,female
3,C0004,Bengaluru,2024-06-21,24.0,female
4,C0005,Bengaluru,2024-10-01,25.0,female


CITY COUNTS
city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184
Name: count, dtype: Int64

GENDER COUNTS
gender
female     610
male       560
unknown     30
Name: count, dtype: int64

MISSING AGES
0

AGE RANGE
18.0 60.0

SIGNUP DATE TYPE
datetime64[us]
Customers: (1200, 5)
Orders: (12309, 11)

ORDER STATUS:
status
completed    11110
cancelled     1199
Name: count, dtype: int64

DUPLICATE ORDER IDs:
0

MISSING ORDER AMOUNTS:
0

MISSING DELIVERY TIMES:
status
cancelled    1199
completed       0
Name: delivery_minutes, dtype: int64

COMPLETED ORDER AMOUNT:
2998668.8


## 4. Query the SQLite database

The clean tables are persisted in `data/instamart.db`. SQLite gives us a portable analytical database that can be queried without setting up a server.

Run the supplied SQL examples and compare SQL totals to pandas results.

In [46]:
import sqlite3
import pandas as pd
from pathlib import Path
db_path = ROOT / "data" / "instamart.db"
clean_dir = ROOT / "data" / "clean"
    
# Work with copies so the original DataFrames remain unchanged
customers_save = customers.copy()
products_save = products.copy()
orders_save = orders.copy()

# Store order_ts as YYYY-MM-DD HH:MM:SS text
orders_save["order_ts"] = (
    pd.to_datetime(orders_save["order_ts"], errors="coerce")
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

# Save SQLite tables
with sqlite3.connect(str(db_path)) as con:
    customers_save.to_sql("customers", con, if_exists="replace", index=False)
    products_save.to_sql("products", con, if_exists="replace", index=False)
    orders_save.to_sql("orders", con, if_exists="replace", index=False)

print("Three CSV files and three SQLite tables saved successfully.")
print("CSV folder:", clean_dir)
print("Database:", db_path)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
display(tables)
q2 = pd.read_sql_query("""
        SELECT c.city, p.category, ROUND(SUM(o.order_amount), 2) AS revenue
        FROM orders o
        JOIN customers c USING(customer_id)
        JOIN products p USING(product_id)
        WHERE o.status = 'completed'
        GROUP BY c.city, p.category
        ORDER BY c.city, p.category
    """, con)
display(q2)

print("SQL grand total INR:", round(q2.revenue.sum(), 2))


Three CSV files and three SQLite tables saved successfully.
CSV folder: c:\Users\HP\Documents\Instamart customer copilot\instamart_customer_copilot_teaching_kit\notebooks\data\clean
Database: c:\Users\HP\Documents\Instamart customer copilot\instamart_customer_copilot_teaching_kit\notebooks\data\instamart.db


,name
0,customers
1,orders
2,products


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


SQL grand total INR: 2998668.8


In [48]:
from pathlib import Path

sql_dir = ROOT / "sql"
sql_dir.mkdir(parents=True, exist_ok=True)

queries = {
"Q1": """
SELECT
    customer_id,
    SUM(order_amount) AS total_large_orders
FROM orders
WHERE status = 'completed'
  AND order_amount > 500
GROUP BY customer_id
HAVING SUM(order_amount) > 8000
ORDER BY total_large_orders DESC;
""",

"Q2": """
SELECT
    c.city,
    p.category,
    ROUND(SUM(o.order_amount), 2) AS revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY c.city, p.category
ORDER BY c.city, p.category;
""",

"Q3": """
SELECT
    product_id,
    product_name,
    category,
    price_inr
FROM products
WHERE price_inr > (
    SELECT AVG(price_inr)
    FROM products
)
ORDER BY price_inr DESC;
""",

"Q4": """
SELECT
    c.customer_id,
    c.city,
    c.gender
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
   AND o.status = 'completed'
WHERE o.order_id IS NULL
ORDER BY c.customer_id;
""",

"Q5": """
WITH customer_spend AS (
    SELECT
        customer_id,
        SUM(order_amount) AS total_spend
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
),
average_spend AS (
    SELECT AVG(total_spend) AS avg_spend
    FROM customer_spend
)
SELECT
    cs.customer_id,
    ROUND(cs.total_spend, 2) AS total_spend,
    ROUND(a.avg_spend, 2) AS average_spend
FROM customer_spend cs
CROSS JOIN average_spend a
WHERE cs.total_spend > a.avg_spend
ORDER BY cs.total_spend DESC;
""",

"Q6": """
WITH customer_spend AS (
    SELECT
        c.customer_id,
        c.city,
        SUM(o.order_amount) AS total_spend
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.customer_id, c.city
),
ranked_customers AS (
    SELECT
        customer_id,
        city,
        ROUND(total_spend, 2) AS total_spend,
        RANK() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS spend_rank,
        DENSE_RANK() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS dense_rank,
        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS row_num
    FROM customer_spend
)
SELECT *
FROM ranked_customers
ORDER BY city, spend_rank, customer_id;
""",

"Q7": """
WITH daily_revenue AS (
    SELECT
        SUBSTR(order_ts, 1, 10) AS day,
        SUM(order_amount) AS revenue
    FROM orders
    WHERE status = 'completed'
    GROUP BY SUBSTR(order_ts, 1, 10)
)
SELECT
    day,
    ROUND(revenue, 2) AS revenue,
    ROUND(
        SUM(revenue) OVER (ORDER BY day),
        2
    ) AS cumulative_revenue
FROM daily_revenue
ORDER BY day;
"""
}

sql_file = sql_dir / "queries.sql"

sql_file.write_text(
    "\n\n".join(
        f"-- {name}\n{query.strip()}"
        for name, query in queries.items()
    ),
    encoding="utf-8"
)
print("Saved:", sql_file)
db_path = ROOT / "data" / "instamart.db"

results = {}

with sqlite3.connect(str(db_path)) as con:
    for name, query in queries.items():
        results[name] = pd.read_sql_query(query, con)
        print(f"\n{name} RESULTS")
        print("Rows:", len(results[name]))
        display(results[name].head(10))
        
with sqlite3.connect(str(db_path)) as con:
    customers_db = pd.read_sql_query("SELECT * FROM customers", con)
    products_db = pd.read_sql_query("SELECT * FROM products", con)
    orders_db = pd.read_sql_query("SELECT * FROM orders", con)

# Join completed orders to customers and products
completed_orders = orders_db[
    orders_db["status"] == "completed"
].copy()

joined = (
    completed_orders
    .merge(
        customers_db[["customer_id", "city"]],
        on="customer_id",
        how="left"
    )
    .merge(
        products_db[["product_id", "category"]],
        on="product_id",
        how="left"
    )
)



Saved: c:\Users\HP\Documents\Instamart customer copilot\instamart_customer_copilot_teaching_kit\notebooks\sql\queries.sql

Q1 RESULTS
Rows: 30


,customer_id,total_large_orders
0,C0693,13776.55
1,C0082,12317.70
2,C0743,12266.85
3,C0525,12241.30
4,C0291,12150.70
5,C0267,12150.25
6,C0488,11825.50
7,C0699,11693.90
8,C0512,11617.65
9,C0332,10779.50



Q2 RESULTS
Rows: 30


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70



Q3 RESULTS
Rows: 11


,product_id,product_name,category,price_inr
0,P038,Bluetooth Speaker,Electronics,1999
1,P033,Wireless Earbuds,Electronics,1499
2,P034,Power Bank 10000mAh,Electronics,1299
3,P036,Phone Charger 20W,Electronics,799
4,P037,Smart Bulb,Electronics,599
5,P040,Memory Card 64GB,Electronics,549
6,P032,Sunscreen 50g,Personal Care,399
7,P030,Body Lotion 400ml,Personal Care,350
8,P039,Phone Stand,Electronics,349
9,P014,Ghee 500ml,Dairy,340


... (4 outputs left out)


Q6 RESULTS
Rows: 1175


,customer_id,city,total_spend,spend_rank,dense_rank,row_num
0,C0082,Bengaluru,12856.20,1,1,1
1,C0525,Bengaluru,12719.70,2,2,2
2,C0699,Bengaluru,12471.80,3,3,3
3,C0543,Bengaluru,11195.10,4,4,4
4,C1147,Bengaluru,10156.90,5,5,5
5,C0407,Bengaluru,9945.85,6,6,6
6,C0942,Bengaluru,8643.60,7,7,7
7,C1126,Bengaluru,8595.70,8,8,8
8,C0916,Bengaluru,8444.85,9,9,9
9,C0098,Bengaluru,8151.20,10,10,10



Q7 RESULTS
Rows: 183


,day,revenue,cumulative_revenue
0,2025-04-01,16876.50,16876.50
1,2025-04-02,19854.55,36731.05
2,2025-04-03,28048.35,64779.40
3,2025-04-04,14249.60,79029.00
4,2025-04-05,16031.50,95060.50
5,2025-04-06,26323.80,121384.30
6,2025-04-07,15606.90,136991.20
7,2025-04-08,17055.45,154046.65
8,2025-04-09,20403.35,174450.00
9,2025-04-10,15359.90,189809.90


In [52]:
import pandas as pd
# Keep the existing category from orders
joined["category"] = joined["category_x"]

# Remove duplicate category columns
joined = joined.drop(
    columns=["category_x", "category_y"],
    errors="ignore"
)

print(joined.columns.tolist())
# Pivot table equivalent of SQL Q2
pivot_q2 = pd.pivot_table(
    joined,
    index="city",
    columns="category",
    values="order_amount",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Grand Total"
).round(2)

display(pivot_q2)

print("SQL total:", round(results["Q2"]["revenue"].sum(), 2))
print("Pandas grand total:", round(joined["order_amount"].sum(), 2))

assert round(joined["order_amount"].sum(), 2) == 2998668.80
city_stats = (
    joined.groupby("city")
    .agg(
        total_revenue=("order_amount", "sum"),
        average_order_amount=("order_amount", "mean"),
        order_count=("order_amount", "count")
    )
    .round(2)
    .sort_index()
)

display(city_stats)
assert round(city_stats.loc["Bengaluru", "total_revenue"], 2) == 523643.40
assert round(city_stats.loc["Bengaluru", "average_order_amount"], 2) == 276.47
assert city_stats.loc["Bengaluru", "order_count"] == 1894

assert round(city_stats.loc["Mumbai", "total_revenue"], 2) == 479058.50
assert round(city_stats.loc["Mumbai", "average_order_amount"], 2) == 297.00
assert city_stats.loc["Mumbai", "order_count"] == 1613

print("City statistics reference checks passed.")
amount = pd.to_numeric(
    completed_orders["order_amount"],
    errors="coerce"
).dropna()

mean_amount = amount.mean()
median_amount = amount.median()
skew_amount = amount.skew()

q1 = amount.quantile(0.25)
q2 = amount.quantile(0.50)
q3 = amount.quantile(0.75)

iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

lower_outliers = amount[amount < lower_fence]
upper_outliers = amount[amount > upper_fence]

print("Mean:", round(mean_amount, 2))
print("Median:", round(median_amount, 2))
print("Skewness:", round(skew_amount, 2))

print("\nQuartiles:")
print("Q1:", round(q1, 2))
print("Q2:", round(q2, 2))
print("Q3:", round(q3, 2))

print("\nIQR:", round(iqr, 2))
print("Lower fence:", round(lower_fence, 2))
print("Upper fence:", round(upper_fence, 2))

print("\nOutliers below lower fence:", len(lower_outliers))
print("Outliers above upper fence:", len(upper_outliers))
correlation_columns = [
    "order_amount",
    "discount_pct",
    "quantity",
    "delivery_minutes"
]

correlation_data = completed_orders[
    correlation_columns
].apply(pd.to_numeric, errors="coerce")

correlation_matrix = correlation_data.corr()

display(correlation_matrix)

# Find the pair with the largest absolute correlation
corr_pairs = correlation_matrix.abs().where(
    ~pd.DataFrame(
        [[i == j for j in correlation_columns]
         for i in correlation_columns],
        index=correlation_columns,
        columns=correlation_columns
    )
).stack()

strongest_pair = corr_pairs.idxmax()

print("Largest absolute correlation pair:", strongest_pair)
print(
    "Correlation value:",
    round(
        correlation_matrix.loc[strongest_pair[0], strongest_pair[1]],
        3
    )
)
correlation_columns = [
    "order_amount",
    "discount_pct",
    "quantity",
    "delivery_minutes"
]

correlation_data = completed_orders[
    correlation_columns
].apply(pd.to_numeric, errors="coerce")

correlation_matrix = correlation_data.corr()

display(correlation_matrix)

# Find the pair with the largest absolute correlation
corr_pairs = correlation_matrix.abs().where(
    ~pd.DataFrame(
        [[i == j for j in correlation_columns]
         for i in correlation_columns],
        index=correlation_columns,
        columns=correlation_columns
    )
).stack()

strongest_pair = corr_pairs.idxmax()

print("Largest absolute correlation pair:", strongest_pair)
print(
    "Correlation value:",
    round(
        correlation_matrix.loc[strongest_pair[0], strongest_pair[1]],
        3
    )
)


['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes', 'price_inr', 'city', 'category']


category,Dairy,Electronics,Grocery,Personal Care,Snacks,Grand Total
city,,,,,,
Bengaluru,87075.05,233909.45,110915.65,20193.85,71549.40,523643.40
Chennai,78301.80,195508.80,93817.45,27675.40,68325.70,463629.15
Delhi,117271.70,158549.10,112658.05,23471.50,90879.45,502829.80
Hyderabad,83731.00,228913.10,109475.40,24800.05,81670.35,528589.90
Mumbai,75209.85,214738.40,78970.15,38018.00,72122.10,479058.50
Pune,90349.55,187629.25,108456.60,34275.15,80207.50,500918.05
Grand Total,531938.95,1219248.10,614293.30,168433.95,464754.50,2998668.80


SQL total: 2998668.8
Pandas grand total: 2998668.8


,total_revenue,average_order_amount,order_count
city,,,
Bengaluru,523643.40,276.47,1894
Chennai,463629.15,272.72,1700
Delhi,502829.80,243.62,2064
Hyderabad,528589.90,284.95,1855
Mumbai,479058.50,297.00,1613
Pune,500918.05,252.48,1984


City statistics reference checks passed.
Mean: 269.91
Median: 135.0
Skewness: 4.61

Quartiles:
Q1: 64.8
Q2: 135.0
Q3: 297.0

IQR: 232.2
Lower fence: -283.5
Upper fence: 645.3

Outliers below lower fence: 0
Outliers above upper fence: 897


,order_amount,discount_pct,quantity,delivery_minutes
order_amount,1.000000,-0.095106,0.087294,-0.003135
discount_pct,-0.095106,1.000000,-0.000359,-0.009808
quantity,0.087294,-0.000359,1.000000,-0.012623
delivery_minutes,-0.003135,-0.009808,-0.012623,1.000000


Largest absolute correlation pair: ('order_amount', 'discount_pct')
Correlation value: -0.095


,order_amount,discount_pct,quantity,delivery_minutes
order_amount,1.000000,-0.095106,0.087294,-0.003135
discount_pct,-0.095106,1.000000,-0.000359,-0.009808
quantity,0.087294,-0.000359,1.000000,-0.012623
delivery_minutes,-0.003135,-0.009808,-0.012623,1.000000


Largest absolute correlation pair: ('order_amount', 'discount_pct')
Correlation value: -0.095


## 5. Part B — customer segments and lapse prediction

RFM means:
- **Recency:** days since the most recent completed order at the snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The model module scales RFM for K-means, compares K values, uses PCA to inspect compressed variance, and builds a lapse model. The lapse model's input features use only orders before 2025-09-01; September is reserved to define the label. The scaler and encoder are fitted inside a training pipeline, not on the test data.

In [16]:
import sys
import os
import pandas as pd
import sqlite3
# 1. Connect to your SQLite database and load the orders table into df
db_path = r"c:\Users\HP\Documents\Instamart customer copilot\instamart_customer_copilot_teaching_kit\notebooks\data\instamart.db"
conn = sqlite3.connect(db_path)
# List all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)["name"].tolist()

for table in tables:
    print(f"\n--- Columns in table: {table} ---")
    cols = pd.read_sql(f"PRAGMA table_info({table});", conn)
    print(cols["name"].tolist())
# Define snapshot date
snapshot = pd.Timestamp("2025-10-01 00:00:00")

# 1. Filter for completed orders
completed_orders = df[df["status"] == "completed"].copy()

# 2. Build the RFM Table
rfm_df = (
    completed_orders.groupby("customer_id")
    .agg(
        recency=("last_order_ts", lambda x: (snapshot - x.max()).days),
        frequency=("order_id", "nunique"),
        monetary=("order_amount", "sum"),
    )
    .reset_index()
)
# Add parent directory to Python path so 'src' can be found
parent_dir = os.path.abspath(os.path.join('..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
from src.customer_models import rfm_and_segments, build_lapse_model
rfm, k_diagnostics, segment_profile, pca_variance = rfm_and_segments(k=4)
display(k_diagnostics.round(3))
display(segment_profile.round(2))
print("PCA explained variance ratio:", pca_variance, "total:", pca_variance.sum())
display(rfm.head())
lapse_summary = build_lapse_model()
lapse_summary



--- Columns in table: customers ---
['customer_id', 'city', 'signup_date', 'age', 'gender']

--- Columns in table: products ---
['product_id', 'product_name', 'category', 'price_inr']

--- Columns in table: orders ---
['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes', 'category', 'price_inr']


KeyError: "Label(s) ['last_order_ts'] do not exist"

### Instructor discussion: leakage
Ask students to identify:
1. Which dates can enter the lapse-model features?
2. Why must September orders not enter the features?
3. Why is a Premium-only retention list labelled *illustrative* if the Premium flag comes from whole-window RFM?
4. Why can accuracy be misleading when a substantial minority of customers lapse?

## 6. Part C — support ticket triage

This pipeline demonstrates:
1. input guardrails for empty and overlong tickets;
2. ordered PII redaction (email → phone → order ID);
3. retry logic for simulated transient errors;
4. strict category validation and a keyword fallback;
5. token budgeting and human-review routing.

The exact user message sent to the mock classifier is the redacted ticket text only.

In [23]:
import sys
from pathlib import Path
project_root = Path().resolve().parent  # Use Path().resolve() if notebook is already in root
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from src.triage import run_pipeline, FEW, ZERO
triage_results, triage_summary = run_pipeline(prompt=FEW, token_budget=100000)
print(triage_summary)
display(triage_results.head(10))
display(triage_results.status.value_counts().rename_axis("status").reset_index(name="tickets"))


retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
{'tokens': 12520, 'redactions': {'email': 55, 'phone': 91, 'order_id': 132}, 'status_counts': {'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9, 'skipped_empty': 4, 'flagged_long': 4}}


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,At Risk,HIGH,security_team,84
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,80
2,T0095,C1087,classified_json,technical,At Risk,NORMAL,support_queue,82
3,T0159,C1172,skipped_empty,,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,At Risk,HIGH,security_team,84
5,T0124,C0139,classified_json,fraud,Frequent,HIGH,security_team,84
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,76
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,79
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,80
9,T0002,C0881,classified_json,delivery,At Risk,NORMAL,support_queue,78


,status,tickets
0,classified_json,135
1,classified_keyword,12
2,flagged_invalid_output,9
3,skipped_empty,4
4,flagged_long,4


### Discussion: privacy and safety
- Redaction is a defence-in-depth step, not a guarantee that every sensitive detail is removed.
- Why are invalid categories sent to human review instead of guessed?
- Why should fraud override ordinary customer-segment routing?
- Why do empty/long tickets need to be blocked before any model call?

## 7. Part D — policy RAG

The retrieval layer chunks policy documents, fits a TF-IDF vectorizer and ranks chunks by cosine similarity. It passes retrieved chunks to the offline policy assistant, which should answer only from the context and fall back when a question is not covered.

In [22]:
import pandas as pd
from src.policy_rag import PolicyRAG, evaluate
rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
print(rag.answer("What delivery time makes a completed order eligible for a full refund?"))
policy_eval = evaluate(rag)
display(policy_eval)
print("Expected-document hits:", int(policy_eval.retrieved_doc.sum()))
print("Expected-fact hits:", int(policy_eval.fact_found.sum()))
print("Fallbacks:", int(policy_eval.fallback.sum()))

Policy chunks: 15


,source,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ..."
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours...."


{'answer': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.', 'sources': ['refund_policy.txt', 'privacy_policy.txt'], 'retrieved': [{'source': 'refund_policy.txt', 'score': 0.35464416504570484, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.31425507733397284, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'privacy_policy.txt', 'score': 0.15759666156595936, 'text': 'Delivery addresses are deleted from active systems 180 days after the last order. A customer can ask for account deletion and the request is c

,question,expected_doc,retrieved_doc,fact_found,fallback
0,What delivery time makes a completed order eli...,refund_policy.txt,True,True,False
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,False
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,False
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,False
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,False
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,False
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,True,False
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,False
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,False


Expected-document hits: 12
Expected-fact hits: 12
Fallbacks: 2


## 8. Part D — refund tools and safe stopping

The refund agent demonstrates a tool loop: look up an order in SQLite, retrieve policy, calculate the eligible refund, then issue it or escalate to a human. In this teaching kit the ledger is a local JSONL file; no money moves.

Before running agent demonstrations, inspect `src/refund_agent.py` and ask students to trace how a tool name maps to a Python function.

In [27]:
from pathlib import Path
import sqlite3
import pandas as pd

db_path = ROOT / "data" / "instamart.db"
from src.refund_agent import get_order, search_policy, calculate_refund, TOOL_MAP, run_agent
print("Tools:", sorted(TOOL_MAP))
print("Example order lookup:")
with sqlite3.connect(db_path) as con:
    example_order = pd.read_sql_query(
        "SELECT order_id FROM orders WHERE status='completed' AND delivery_minutes > 45 LIMIT 1", con
    ).iloc[0, 0]
print(example_order, get_order(example_order))
print("Refund calculation:", calculate_refund(250.0, 52))
print("Timeout safety demo:", run_agent(f"Refund order {example_order}", scenario_id="timeout-demo", time_limit_seconds=0))

Tools: ['calculate_refund', 'escalate_to_human', 'get_order', 'issue_refund', 'search_policy']
Example order lookup:
ORD-100048 {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65}
Refund calculation: {'refund_inr': 250.0}
Timeout safety demo: {'status': 'FALLBACK_TIMEOUT', 'final_answer': 'FALLBACK_TIMEOUT: hand off to a human agent.', 'steps': [], 'llm_calls': 0, 'tokens_used': 0}


## 9. Final generated artifacts and reproducibility check

Review the CSVs and model file generated by the pipeline. In class, have students delete `data/clean/`, `data/instamart.db`, and `outputs/`, then rerun the documented commands to verify that the project can be rebuilt from source.

**Important:** This teaching kit is an instructional implementation. If the course grades exact numeric checkpoints from the original brief, run every checkpoint and reconcile differences; do not hardcode expected answers.

In [28]:
db_path = ROOT / "data" / "instamart.db"
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())

\ndata\clean
['customers.csv', 'orders.csv', 'products.csv']
\noutputs
['customer_segments.csv', 'lapse_model.joblib', 'retention_list.csv', 'triage_results.csv']
\nstate
[]
Database exists: True
